<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 00 — Environment Check

**Deep Learning for Engineering · Aalborg University · Part 1**

Run this notebook first, top to bottom. There is nothing to write in it. Its job
is to establish, before you spend an hour on anything else, that `torch`,
`numpy` and `matplotlib` are importable, that `Ex_6_core.py` imports cleanly,
that the four datasets can be generated on **your** machine with no network
connection, and that the four optimisers this exercise set compares all run —
including **L-BFGS**, which has a different calling convention from the others
and which every exercise in Part 2 uses.

If a cell fails here, fix it before going on.

---

## 0 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

**What you should see.** Four version numbers and `cuda available: False` on
most machines. Any Python from 3.9 and any PyTorch from 2.0 will do. **No GPU is
required anywhere in Ex_06.**

---

## 1 · The shared module and its four datasets

In [ ]:
import Ex_6_core as core

print("module file:", core.__file__)
print("output dir :", core.OUTPUT_DIR)
print()

x_cal, y_cal = core.calibration_dataset()
X_vib, y_vib = core.vibration_dataset()
x_res, y_res = core.response_dataset()
x_fat, y_fat = core.fatigue_dataset()
x_val, y_val = core.fatigue_validation()

print("calibration line   :", x_cal.shape, "  true slope", core.TRUE_SLOPE,
      " intercept", core.TRUE_INTERCEPT, " sigma", core.TRUE_SIGMA)
print("vibration classes  :", X_vib.shape, " counts", np.bincount(y_vib),
      " ", core.VIBRATION_CLASSES)
print("damped response    :", x_res.shape, " noiseless:",
      bool(np.allclose(y_res, core.damped_response(x_res))))
print("fatigue, training  :", x_fat.shape, " held out:", x_val.shape)

**What you should see.**

```
calibration line   : (40,)   true slope 2.4  intercept 0.8  sigma 0.35
vibration classes  : (360, 2)  counts [120 120 120]   ('balanced', 'imbalance', 'bearing fault')
damped response    : (200,)  noiseless: True
fatigue, training  : (20,)  held out: (60,)
```

Note `noiseless: True` for the damped response. That is deliberate and notebook
02 explains it: when you are comparing optimisers, noisy data makes every
optimiser stop at the same floor and the comparison measures nothing.

---

## 2 · Look at all four

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.0, 7.4))

axes[0, 0].plot(x_cal, y_cal, "o", ms=5, color="#111111")
axes[0, 0].plot(x_cal, core.TRUE_SLOPE * x_cal + core.TRUE_INTERCEPT, lw=1.6,
                ls="--", color="#888888")
axes[0, 0].set_title("calibration: reading against load")
axes[0, 0].set_xlabel("load [normalised]"); axes[0, 0].set_ylabel("reading")

core.plot_classes(X_vib, y_vib, ax=axes[0, 1], title="vibration classes")

axes[1, 0].plot(x_res, y_res, lw=1.8, color="#1f77b4")
axes[1, 0].set_title("damped structural response (noiseless)")
axes[1, 0].set_xlabel("$x$"); axes[1, 0].set_ylabel("$y$")

grid = np.linspace(0, 1, 300)
axes[1, 1].plot(grid, core.fatigue_truth(grid), lw=1.5, ls="--",
                color="#888888", label="truth")
axes[1, 1].plot(x_val, y_val, "s", ms=4, mfc="none", mec="#0f9d58",
                label="held out (60)")
axes[1, 1].plot(x_fat, y_fat, "o", ms=7, color="#111111", label="training (20)")
axes[1, 1].set_title("fatigue measurements")
axes[1, 1].legend(frameon=False, fontsize=8)
for a in axes.ravel():
    a.grid(alpha=0.25)
fig.tight_layout()
plt.show()

**What you should see.** Four panels: a noisy straight line; three
overlapping clouds of points; a smooth decaying oscillation with about two and a
quarter cycles; and twenty black training points with sixty green held-out
squares around a dashed truth curve.

Two of these deserve a second look now, because the notebooks that use them
depend on you having noticed.

**The vibration classes overlap.** That is not sloppy generation. A perfectly
separable problem hides everything interesting about a loss function, because
every loss agrees when the answer is obvious.

**There are twenty fatigue points and sixty held out.** Twenty is realistic —
each point is a specimen taken to failure, and specimens cost days. Sixty held
out is not realistic at all, and notebook 03 says so: it exists only because the
data is synthetic, and it is the only thing that can tell you the flexible model
is worse.

---

## 3 · The four optimisers

Three of them share a calling convention. The fourth does not, and the fourth is
the one Part 2 depends on.

In [ ]:
core.set_seed(0)
x, y = core.response_dataset(n=60)
X, Y = core.to_tensor(x), core.to_tensor(y)
loss_fn = nn.MSELoss()

for name, make in [
        ("SGD           ", lambda p: torch.optim.SGD(p, lr=0.05)),
        ("SGD + momentum", lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
        ("Adam          ", lambda p: torch.optim.Adam(p, lr=0.01))]:
    core.set_seed(0)
    model = core.MLP(hidden=(16, 16))
    opt = make(model.parameters())
    for _ in range(200):
        opt.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        opt.step()
    print(f"{name}  loss after 200 steps: {loss.item():.5f}")

# L-BFGS: same idea, different interface. It may evaluate the loss several
# times per step, so it needs a function it can call rather than a single
# backward pass. That function is the "closure".
core.set_seed(0)
model = core.MLP(hidden=(16, 16))
opt = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=20)

def closure():
    opt.zero_grad()
    loss = loss_fn(model(X), Y)
    loss.backward()
    return loss

for _ in range(10):
    opt.step(closure)
with torch.no_grad():
    print(f"L-BFGS          loss after 10 steps: "
          f"{loss_fn(model(X), Y).item():.5f}")

**What you should see.** Four losses, all finite, with L-BFGS well below the
other three. The exact numbers depend on your PyTorch version.

The important thing here is not the numbers, it is the **shape of the L-BFGS
call**. Look at it again:

```python
def closure():
    opt.zero_grad()
    loss = loss_fn(model(X), Y)
    loss.backward()
    return loss

opt.step(closure)
```

The closure does the whole forward-and-backward, and returns the loss.
`opt.step` may call it several times inside a single step, because a quasi-Newton
method performs a **line search**: it decides on a direction, then tries
different distances along it and evaluates the loss at each. The other three
optimisers take a fixed step and never look back, so they need only one
evaluation.

Two consequences follow, and notebook 02 measures both.

**"One step" means different amounts of work for different optimisers.** Any
plot of loss against step number is comparing unequal things, and must say so.

**L-BFGS is full batch.** The line search compares losses, and a loss computed on
a different mini-batch each time is not comparable. This is why L-BFGS appears in
physics-informed work, where the whole problem fits in memory, and essentially
never in large-scale training.

---

## 4 · Ready

If every cell above ran, you are set up. Continue with
**`Ex06_01_loss_functions.ipynb`**.

A note on time. Notebooks 00, 01 and 04 take well under a minute of compute each.
Notebook 02 trains four models for three thousand steps and takes a minute or
two. Notebook 03 is the longest: it trains eleven networks for fifteen thousand
epochs each and takes a few minutes. Nothing needs a GPU.